<a href="https://colab.research.google.com/github/OwenLoughery/Project3-Wine_Reviews/blob/main/4401_Project_3_Model_Stanley.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Classification between two similar wines

##Setup: Finding similar wines in pairs

In [9]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
df = pd.read_csv("winemag-data-130k-v2.csv")

wine_data = df.dropna(
    subset=["variety", "description"]
).copy()

variety_counts = wine_data["variety"].value_counts()

valid_varieties = variety_counts[
    variety_counts >= 200
].index

wine_data = wine_data[
    wine_data["variety"].isin(valid_varieties)
].copy()

print("Varieties:", wine_data["variety"].nunique())
print("Reviews:", len(wine_data))

Varieties: 62
Reviews: 118263


In [11]:
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    max_features=10000,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(
    wine_data["description"]
)

print("TF-IDF matrix shape:", X_tfidf.shape)

TF-IDF matrix shape: (118263, 10000)


In [12]:
varieties = sorted(wine_data["variety"].unique())

variety_vectors = []

for variety in varieties:

    mask = (
        wine_data["variety"] == variety
    ).to_numpy()

    mean_vector = np.asarray(
        X_tfidf[mask].mean(axis=0)
    ).ravel()

    variety_vectors.append(mean_vector)

variety_vectors = np.vstack(variety_vectors)

In [13]:
similarity_matrix = cosine_similarity(variety_vectors)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=varieties,
    columns=varieties
)

# Extract unique pairs without self-comparisons
upper_i, upper_j = np.triu_indices(
    len(varieties), k=1
)

similar_pairs = pd.DataFrame({
    "Wine 1": np.array(varieties)[upper_i],
    "Wine 2": np.array(varieties)[upper_j],
    "Similarity": similarity_matrix[upper_i, upper_j]
})

similar_pairs = (
    similar_pairs
    .sort_values("Similarity", ascending=False)
    .reset_index(drop=True)
)

display(similar_pairs.head(15))

,Wine 1,Wine 2,Similarity
0,Tempranillo,Tempranillo Blend,0.950381
1,Nebbiolo,Sangiovese,0.891038
2,Malbec,Tempranillo,0.878627
3,Cabernet Sauvignon,Merlot,0.864705
4,Gamay,Portuguese Red,0.837147
5,Malbec,Tempranillo Blend,0.832999
6,Garnacha,Tempranillo,0.829374
7,Cabernet Franc,Cabernet Sauvignon,0.827344
8,Bordeaux-style White Blend,Portuguese White,0.825607
9,Bordeaux-style Red Blend,Portuguese Red,0.821633


In [14]:
similar_pairs["Wine 1 Count"] = (
    similar_pairs["Wine 1"].map(variety_counts)
)

similar_pairs["Wine 2 Count"] = (
    similar_pairs["Wine 2"].map(variety_counts)
)

display(similar_pairs.head(15))

,Wine 1,Wine 2,Similarity,Wine 1 Count,Wine 2 Count
0,Tempranillo,Tempranillo Blend,0.950381,1810,588
1,Nebbiolo,Sangiovese,0.891038,2804,2707
2,Malbec,Tempranillo,0.878627,2652,1810
3,Cabernet Sauvignon,Merlot,0.864705,9472,3102
4,Gamay,Portuguese Red,0.837147,1025,2466
5,Malbec,Tempranillo Blend,0.832999,2652,588
6,Garnacha,Tempranillo,0.829374,326,1810
7,Cabernet Franc,Cabernet Sauvignon,0.827344,1353,9472
8,Bordeaux-style White Blend,Portuguese White,0.825607,1066,1159
9,Bordeaux-style Red Blend,Portuguese Red,0.821633,6915,2466


##Model Building

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [16]:
pair_df = df[
    df["variety"].isin(["Nebbiolo", "Sangiovese"])
].copy()

pair_df = pair_df.dropna(
    subset=["variety", "description"]
)

pair_df = pair_df[
    pair_df["description"].str.strip() != ""
].copy()

# Remove exact duplicate descriptions to reduce leakage
# Conflicting labels for identical text should be investigated
conflicts = (
    pair_df.groupby("description")["variety"].nunique()
)

ambiguous = conflicts[conflicts > 1].index

print("Conflicting descriptions:", len(ambiguous))

pair_df = pair_df[
    ~pair_df["description"].isin(ambiguous)
].drop_duplicates(subset=["description"]).copy()

X = pair_df["description"]
y = pair_df["variety"]

# 80% development, 20% final test
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 80% of development for training, 20% for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev,
    test_size=0.20,
    random_state=42,
    stratify=y_dev
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Conflicting descriptions: 0
Training: 3239
Validation: 810
Test: 1013


In [17]:
model_configs = {
    "Simple": {
        "max_features": 100,
        "ngram_range": (1, 1),
        "C": 1.0
    },
    "Moderate": {
        "max_features": 1000,
        "ngram_range": (1, 1),
        "C": 1.0
    },
    "Complex": {
        "max_features": 5000,
        "ngram_range": (1, 2),
        "C": 10.0
    }
}

def create_model(config):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=config["max_features"],
            ngram_range=config["ngram_range"],
            min_df=1,
            stop_words="english"
        )),
        ("clf", LogisticRegression(
            C=config["C"],
            max_iter=2000,
            random_state=42
        ))
    ])